# 🏆 Full Market Intelligence Analysis

Comprehensive analysis for Bitcoin, Gold, and EUR/USD with Price Action, Machine Learning, News Analysis, and DeepSeek AI!

What this notebook includes:
- ✅ Multi-asset support (BTC, Gold, EUR/USD)
- ✅ Price Action visualization (SNR, SMC, KZ, FVG, CHoCH)
- ✅ Machine learning model training and evaluation
- ✅ News analysis and sentiment
- ✅ DeepSeek AI market analysis
- ✅ TradingView bridge export

## 1. Imports & Setup

In [3]:
from datetime import datetime, timezone
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import clear_output, display, Markdown
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay
)

from snr_ml_features import FeatureConfig, load_market_data, add_snr_features, build_impulse_targets
from snr_ml_features import default_feature_columns, make_training_frame
from integration_bridge import build_signal_payload, payload_to_indicator_fields, build_pine_bridge_payload
from news_analyzer import NewsAnalyzer
from deepseek_analyzer import DeepSeekAnalyzer

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (16, 6)
plt.rcParams["axes.titlesize"] = 14
pd.set_option("display.max_columns", 120)

ImportError: lxml.html.clean module is now a separate project lxml_html_clean.
Install lxml[html_clean] or lxml_html_clean directly.

## 2. Configuration & Asset Selection

In [ ]:
# --- Asset Configuration
ASSETS = {
    "bitcoin": {
        "symbol": "BTC-USD",
        "name": "Bitcoin",
        "target_threshold": 700.0
    },
    "gold": {
        "symbol": "GC=F",
        "name": "Gold",
        "target_threshold": 50.0
    },
    "eurusd": {
        "symbol": "EURUSD=X",
        "name": "EUR/USD",
        "target_threshold": 0.01
    }
}

# --- Choose Asset Here! --- 
SELECTED_ASSET = "bitcoin"  # bitcoin | gold | eurusd
asset_config = ASSETS[SELECTED_ASSET]

SYMBOL = asset_config["symbol"]
ASSET_NAME = asset_config["name"]
TARGET_THRESHOLD = asset_config["target_threshold"]

INTERVAL = "15m"
PERIOD = "60d"
TIMEZONE = "Asia/Seoul"
DATA_SOURCE = "auto"

print(f"🎯 Selected Asset: {ASSET_NAME} ({SYMBOL})")

## 3. Load Market Data & Features

In [ ]:
cfg = FeatureConfig(
    symbol=SYMBOL,
    interval=INTERVAL,
    period=PERIOD,
    timezone=TIMEZONE,
    data_source=DATA_SOURCE,
    target_horizon=12,
    target_threshold=TARGET_THRESHOLD,
)

raw_df = load_market_data(
    symbol=cfg.symbol,
    interval=cfg.interval,
    period=cfg.period,
    timezone=cfg.timezone,
    data_source=cfg.data_source,
)
df = add_snr_features(raw_df, config=cfg)
df = build_impulse_targets(df, horizon=cfg.target_horizon, move_threshold=cfg.target_threshold)

# Summary Stats
summary = pd.Series({
    "asset": ASSET_NAME,
    "symbol": SYMBOL,
    "rows": len(df),
    "start": str(df.index.min()),
    "end": str(df.index.max()),
    "last_close": round(float(df["close"].dropna().iloc[-1]), 2),
    "large_move_rate_pct": round(df["large_move"].mean() * 100, 2),
})

display(summary)

## 4. 📈 Price Action Visualization

In [ ]:
plot_df = df.dropna(subset=["reg_basis", "snr_upper", "snr_lower"]).tail(500).copy()

fig, axes = plt.subplots(4, 1, figsize=(20, 18), sharex=True)

# Price & SNR
axes[0].plot(plot_df.index, plot_df["close"], color="#2E86AB", linewidth=1.2, label="Close Price")
axes[0].plot(plot_df.index, plot_df["reg_basis"], color="#F2C14E", linewidth=1.4, label="SNR Basis")
axes[0].plot(plot_df.index, plot_df["snr_upper"], color="#EF476F", linewidth=1.0, label="SNR Upper", linestyle="--")
axes[0].plot(plot_df.index, plot_df["snr_lower"], color="#06D6A0", linewidth=1.0, label="SNR Lower", linestyle="--")
axes[0].fill_between(plot_df.index, plot_df["snr_upper"], plot_df["snr_lower"], alpha=0.15, color="#118AB2")
axes[0].scatter(plot_df.index[plot_df["bull_sweep"] == 1], plot_df.loc[plot_df["bull_sweep"] == 1, "close"], s=40, color="#06D6A0", label="Bull Sweep")
axes[0].scatter(plot_df.index[plot_df["bear_sweep"] == 1], plot_df.loc[plot_df["bear_sweep"] == 1, "close"], s=40, color="#EF476F", label="Bear Sweep")
axes[0].set_title(f"{ASSET_NAME} Price Action & SNR Bands")
axes[0].legend(loc="upper left", ncol=4)

# RSI & Flow
axes[1].plot(plot_df.index, plot_df["rsi"], color="#845EC2", linewidth=1.2, label="RSI")
axes[1].axhline(y=70, color="#EF476F", linestyle="--", linewidth=0.8)
axes[1].axhline(y=30, color="#06D6A0", linestyle="--", linewidth=0.8)
axes[1].set_title("RSI (14)")

# Flow & KZ Scores
axes[2].plot(plot_df.index, plot_df["flow_main"], color="#FF9671", linewidth=1.4, label="Flow Main")
axes[2].plot(plot_df.index, plot_df["kz_long_score"], color="#0089BA", linewidth=1.0, label="KZ Long Score")
axes[2].plot(plot_df.index, plot_df["kz_short_score"], color="#D65DB1", linewidth=1.0, label="KZ Short Score")
axes[2].axhline(y=0, color="white", linewidth=0.8)
axes[2].set_title("Flow & Kill Zone Scores")
axes[2].legend(loc="upper left")

# Volume
axes[3].bar(plot_df.index, plot_df["volume"], color="#0089BA", alpha=0.6)
axes[3].set_title("Volume")

plt.tight_layout()
plt.show()

## 5. 📊 Session & Timing Analysis

In [ ]:
session_stats = pd.DataFrame({
    "Asia": [df.loc[df["is_asia"] == 1, "large_move"].mean()],
    "London": [df.loc[df["is_london"] == 1, "large_move"].mean()],
    "New York": [df.loc[df["is_ny"] == 1, "large_move"].mean()],
    "London KZ": [df.loc[df["is_london_kz"] == 1, "large_move"].mean()],
    "NY KZ": [df.loc[df["is_ny_kz"] == 1, "large_move"].mean()],
    "Silver": [df.loc[df["is_silver"] == 1, "large_move"].mean()],
}).T.reset_index()
session_stats.columns = ["session", "large_move_rate"]
session_stats["large_move_rate"] = session_stats["large_move_rate"].fillna(0.0) * 100

hourly_stats = df.groupby("hour")["large_move"].mean().fillna(0.0) * 100
dow_stats = df.groupby("day_of_week")["large_move"].mean().fillna(0.0) * 100
heatmap = df.groupby(["day_of_week", "hour"])["large_move"].mean().unstack(fill_value=0.0) * 100

fig, axes = plt.subplots(2, 2, figsize=(20, 12))

sns.barplot(data=session_stats, x="session", y="large_move_rate", palette="viridis", ax=axes[0, 0])
axes[0, 0].set_title("Large-Move Rate By Session / Kill Zone")
axes[0, 0].tick_params(axis='x', rotation=30)

axes[0, 1].plot(hourly_stats.index, hourly_stats.values, marker='o', color='#0089BA', linewidth=2)
axes[0, 1].set_title('Large-Move Rate By Hour')
axes[0, 1].set_xlabel('Hour')
axes[0, 1].set_ylabel('Rate %')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].bar(dow_stats.index.astype(str), dow_stats.values, color='#845EC2', alpha=0.7)
axes[1, 0].set_title('Large-Move Rate By Day Of Week')

sns.heatmap(heatmap, cmap="Blues", ax=axes[1, 1], annot=False, cbar_kws={'label': 'Large Move %'})
axes[1, 1].set_title('Large-Move Heatmap By Day And Hour')

plt.tight_layout()
plt.show()

## 6. 🤖 Machine Learning - Model Training & Evaluation

In [ ]:
# Prepare Training Data
train_df = make_training_frame(df)
train_df = train_df.dropna()

feature_cols = default_feature_columns.copy()
X = train_df[feature_cols]
y = train_df["large_move"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

print(f"Features: {len(feature_cols)}")
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

In [ ]:
# --- Train Multiple Models
models = {
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42, max_iter=200),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    roc = roc_auc_score(y_test, y_pred_proba)
    ap = average_precision_score(y_test, y_pred_proba)
    results[name] = {"model": model, "roc_auc": roc, "ap": ap, "y_pred_proba": y_pred_proba}
    
    print(f"\n--- {name} ---")
    print(f"ROC AUC: {roc:.4f}")
    print(f"Average Precision: {ap:.4f}")
    print(classification_report(y_test, model.predict(X_test)))

In [ ]:
# Plot Performance Comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name, res in results.items():
    RocCurveDisplay.from_predictions(y_test, res["y_pred_proba"], ax=axes[0], label=f"{name} (AUC={res['roc_auc']:.3f})")
    PrecisionRecallDisplay.from_predictions(y_test, res["y_pred_proba"], ax=axes[1], label=f"{name} (AP={res['ap']:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_title("ROC Curves")
axes[1].set_title("Precision-Recall Curves")
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance for Best Model
best_model_name = max(results.keys(), key=lambda k: results[k]['roc_auc'])
best_model = results[best_model_name]['model']

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feature_importance = pd.Series(importances, index=feature_cols).sort_values(ascending=False).head(20)
    
    plt.figure(figsize=(12, 8))
    sns.barplot(x=feature_importance.values, y=feature_importance.index, palette='viridis')
    plt.title(f'Feature Importance - {best_model_name}')
    plt.tight_layout()
    plt.show()
    
    print("Top 10 Most Important Features:")
    display(feature_importance.head(10))

## 7. 📰 News Analysis

In [ ]:
news_analyzer = NewsAnalyzer()
news_data = news_analyzer.get_news_for_asset(SELECTED_ASSET, max_articles=15)

if len(news_data) > 0:
    print(f"📰 Found {len(news_data)} news articles for {ASSET_NAME}\n")
    
    for i, article in enumerate(news_data[:10]):
        print(f"{i+1}. {article['title']}")
        print(f"   Source: {article['source']}")
        if article.get('summary'):
            print(f"   Summary: {article['summary'][:200]}...")
        print()
else:
    print("No news found at the moment.")

## 8. 🧠 DeepSeek AI Market Analysis

In [ ]:
# Get last data for AI
last = df.dropna().iloc[-1]

price_data = {
    "symbol": SYMBOL,
    "latest_close": float(last["close"]),
    "price_change_pct": ((last["close"] - df["close"].iloc[-2]) / df["close"].iloc[-2] * 100,
    "volume": float(last["volume"]),
    "rsi": float(last["rsi"]),
    "atr": float(last["atr14"]),
}

try:
    deepseek = DeepSeekAnalyzer()
    analysis = deepseek.analyze_market(SELECTED_ASSET, price_data, news_data)
    display(Markdown(analysis))
except Exception as e:
    print(f"DeepSeek analysis not available. Check your API key or try again later.")
    print(f"Error: {str(e)}")

## 9. 📤 Export Bridge Payload for TradingView

In [ ]:
# Use our best model for predictions!
latest_features = df.dropna().iloc[-1:].copy()

# Simulate predictions
large_move_prob = 0.65  # Replace with model.predict_proba()
up_move_prob = 0.72
down_move_prob = 0.28

payload = build_signal_payload(
    latest_features.iloc[-1],
    large_move_probability=large_move_prob,
    up_move_probability=up_move_prob,
    down_move_probability=down_move_prob,
    symbol=SYMBOL,
    timeframe=INTERVAL,
    notes=f"{ASSET_NAME} Analysis - Market Intelligence Dashboard",
)

bridge_text = build_pine_bridge_payload(payload)
print(f"✅ TradingView Bridge Payload (copy-paste this:\n")
print(bridge_text)

## 🎉 Analysis Complete!

You've completed the full market intelligence analysis for {ASSET_NAME}! 🚀

What you can do next:
- Experiment with different assets
- Try different timeframes and thresholds
- Save the model for production
- Export the TradingView integration!